# Steam 최고 인기 게임 리뷰 데이터 추출기

이 노트북은 Steam 최고 인기 게임 목록 순위를 기반으로 각 게임당 리뷰 1000개씩을 추출합니다.
- **추출 리뷰 필터 조건**: 한국어 리뷰, 중복 제거, 비정상적 줄바꿈 및 다중 공백 제거, 가독성이 있는 리뷰만(한글 10자 이상 필터 적용).
- **이어서 추출(Checkpoint) 지원**: 중간에 오류가 나거나 수동 중지하더라도 `scraper_checkpoint.json` 파일을 기반으로 이어서 실행합니다.
- 최종 파일은 지정된 `popgame.csv`와 `popgame.json` 두 가지 확장자로 저장됩니다.

In [12]:
!pip install requests pandas bs4 tqdm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import os
import json
import time
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

In [14]:
def get_top_seller_app_ids(max_games=4356):
    app_ids = []
    print("🌟 최고 인기 게임 App ID를 수집 중입니다...")
    url = "https://store.steampowered.com/search/results"
    session = requests.Session()
    start = 0
    pbar = tqdm(total=max_games, desc="게임 고유 ID 수집")
    
    while len(app_ids) < max_games:
        params = {'filter': 'topsellers', 'start': start, 'count': 100, 'infinite': 1}
        try:
            response = session.get(url, params=params)
            data = response.json()
            if not data.get('results_html'): break
                
            soup = BeautifulSoup(data['results_html'], 'html.parser')
            games = soup.find_all('a', href=True)
            
            new_ids = []
            for game in games:
                app_id = game.get('data-ds-appid')
                if app_id:
                    for aid in app_id.split(','):
                        if aid not in app_ids and aid not in new_ids:
                            new_ids.append(aid)
            
            if not new_ids: break
                
            space_left = max_games - len(app_ids)
            added = new_ids[:space_left]
            app_ids.extend(added)
            pbar.update(len(added))
            start += 50
            time.sleep(0.5)
        except Exception as e:
            print(f"ID 추출 중 오류 발생: {e}")
            break
    pbar.close()
    return app_ids

In [16]:
def get_app_details(app_id):
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}&l=korean"
    try:
        res = requests.get(url, timeout=10)
        data = res.json()
        if data and str(app_id) in data and data[str(app_id)]['success']:
            details = data[str(app_id)]['data']
            name = details.get('name', 'Unknown')
            genres = [g['description'] for g in details.get('genres', [])]
            return name, ", ".join(genres)
    except: pass
    return "Unknown", "Unknown"

def clean_review_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'[\r\n\t\v\f]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def is_legible_korean(text):
    if len(text) < 10: return False
    if not re.search(r'[가-힣]', text): return False
    return True

In [17]:
def get_game_reviews(app_id, game_name, game_genre, target_count=1000):
    reviews_data = []
    cursor = '*'
    url = f"https://store.steampowered.com/appreviews/{app_id}"
    
    while len(reviews_data) < target_count:
        params = {'json': 1, 'filter': 'updated', 'language': 'koreana', 'day_range': '9223372036854775807', 'cursor': cursor, 'review_type': 'all', 'purchase_type': 'all', 'num_per_page': 100}
        try:
            res = requests.get(url, params=params, timeout=10)
            data = res.json()
            if data.get('success', 0) != 1 or not data.get('reviews'): break
                
            new_cursor = data['cursor']
            if new_cursor == cursor: break
            cursor = new_cursor
            
            for r in data['reviews']:
                text = clean_review_text(r['review'])
                if is_legible_korean(text):
                    review_item = {
                        '게임명': game_name, '장르': game_genre, '추천여부': '추천' if r['voted_up'] else '비추천',
                        '작성 시점 플레이타임 (분)': r['author'].get('playtime_at_review', 0),
                        '리뷰 원문': text, '분석된 속성': None, 'review_id': r['recommendationid']
                    }
                    reviews_data.append(review_item)
                    if len(reviews_data) >= target_count: break
            time.sleep(1)
        except Exception as e:
            print(f"[{app_id}] 리뷰 추출 중 오류: {e}")
            break
    return reviews_data

In [18]:
def run_review_scraper(max_games=4356, reviews_per_game=1000, csv_filename='popgame_csv.csv', json_filename='popgame_json.json'):
    checkpoint_file = 'scraper_checkpoint.json'
    processed_apps, all_reviews = [], []
    
    if os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, 'r', encoding='utf-8') as f:
                chk_data = json.load(f)
                processed_apps = chk_data.get('processed_apps', [])
                all_reviews = chk_data.get('reviews', [])
            print(f"🔄 [체크포인트] 기존 게임 {len(processed_apps)}개 | 리뷰 {len(all_reviews)}건 이어서 시작합니다.")
        except json.JSONDecodeError:
            print("⚠️ 체크포인트 파일 오류로 처음부터 시작합니다.")
    
    app_ids = get_top_seller_app_ids(max_games)
    
    try:
        for app_id in tqdm(app_ids, desc="게임 리뷰 추출 진행도"):
            if app_id in processed_apps: continue
            game_name, game_genre = get_app_details(app_id)
            if game_name == "Unknown":
                time.sleep(1)
                continue
                
            game_reviews = get_game_reviews(app_id, game_name, game_genre, reviews_per_game)
            all_reviews.extend(game_reviews)
            processed_apps.append(app_id)
            
            with open(checkpoint_file, 'w', encoding='utf-8') as f:
                json.dump({'processed_apps': processed_apps, 'reviews': all_reviews}, f, ensure_ascii=False)
    except KeyboardInterrupt:
        print("\n🛑 중단됨. 데이터를 저장합니다.")
        
    if all_reviews:
        df = pd.DataFrame(all_reviews)
        df.drop_duplicates(subset=['review_id'], inplace=True)
        columns_to_keep = ['게임명', '장르', '추천여부', '작성 시점 플레이타임 (분)', '리뷰 원문', '분석된 속성']
        df = df[columns_to_keep]
        
        df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        df.to_json(json_filename, orient='records', force_ascii=False, indent=4)
        
        print("\n" + "="*50)
        print(f"✅ 추출 완료! 총 {len(df)}건 리뷰 저장: {csv_filename}, {json_filename}")
        print("="*50)
        return df
    return None

In [19]:
# 추출 실행
df_result = run_review_scraper(max_games=4356, reviews_per_game=1000)
if df_result is not None:
    display(df_result.head())

🌟 최고 인기 게임 App ID를 수집 중입니다...


게임 고유 ID 수집:   0%|          | 0/4356 [00:00<?, ?it/s]

게임 리뷰 추출 진행도:   0%|          | 0/100 [00:00<?, ?it/s]


✅ 추출 완료! 총 24045건 리뷰 저장: popgame_csv.csv, popgame_json.json


,게임명,장르,추천여부,작성 시점 플레이타임 (분),리뷰 원문,분석된 속성
0,PUBG: BATTLEGROUNDS,"액션, 어드벤처, 대규모 멀티플레이어, 무료 플레이",비추천,982,이전은 선행해 결정된 이벤트라 넘어갔더니 카2나를 재출시하네 퉤ㅗㅗ,None
1,PUBG: BATTLEGROUNDS,"액션, 어드벤처, 대규모 멀티플레이어, 무료 플레이",비추천,63158,핵이 진짜 너무 역겨울정도로 많음 경쟁가면 기본적으로 ESP는 있는거같음 제발 쓸데...,None
2,PUBG: BATTLEGROUNDS,"액션, 어드벤처, 대규모 멀티플레이어, 무료 플레이",비추천,64740,썅놈들아 최적화 패치좀 해라,None
3,PUBG: BATTLEGROUNDS,"액션, 어드벤처, 대규모 멀티플레이어, 무료 플레이",비추천,12553,실행이 잘 안됨..,None
4,PUBG: BATTLEGROUNDS,"액션, 어드벤처, 대규모 멀티플레이어, 무료 플레이",추천,15855,핵 좀 잡바라 시ba,None
